# 05 — 变分推断与 VAE
生成模型仍然为 $p(x)=N(0,1)$、$p_\theta(y|x)=N(f_\theta(x),\sigma^2I)$。新增 $q_\phi(x|y)=N(m_\phi(y),s_\phi(y)^2)$，由编码器定义。它不是模型的真实后验。

$$\log p_\theta(y)=\underbrace{\mathbb E_q[\log p_\theta(y|x)]-D_{KL}(q\|p(x))}_{ELBO}+D_{KL}(q\|p_\theta(x|y)).$$
先固定生成模型，仅优化近似后验（变分推断）；再同时优化两个网络（VAE）。若上一章已运行，读取其生成模型。否则使用下面的简短网格训练构建相同模型。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm, multivariate_normal
rng = np.random.default_rng(42)
# 仅用于制造观测数据；训练代码不使用真实类别或真实参数。
centers = np.array([[-2., 0.], [2., 0.], [0., 3.]])
y = centers[rng.integers(0, 3, 900)] + 0.45*rng.normal(size=(900, 2))
def compare(samples):
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
    for a, data, title in zip(ax, [y, samples], ['Observed', 'Generated']):
        a.scatter(*data.T, s=5, alpha=.5); a.set_title(title); a.set_aspect('equal')
    plt.show()


In [ ]:
import torch
from torch import nn
from pathlib import Path
torch.set_num_threads(1); torch.manual_seed(42)
Y=torch.tensor(y,dtype=torch.float32); sigma=.45
def make_decoder():
    return nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,2))
decoder=make_decoder()
if Path('decoder_grid.pt').exists():
    decoder.load_state_dict(torch.load('decoder_grid.pt',weights_only=True))
else:
    g=torch.linspace(-6,6,241)[:,None]
    lp=torch.distributions.Normal(0.,1.).log_prob(g[:,0])
    opt=torch.optim.Adam(decoder.parameters(),lr=.01)
    for _ in range(350):
        obs=Y[torch.randint(len(Y),(128,))]
        lj=torch.distributions.Normal(decoder(g),sigma).log_prob(obs[:,None,:]).sum(-1)+lp
        loss=-(torch.logsumexp(lj,1)+torch.log(g[1,0]-g[0,0])).mean()
        opt.zero_grad(); loss.backward(); opt.step()
encoder=nn.Sequential(nn.Linear(2,32),nn.Tanh(),nn.Linear(32,2))
def posterior_params(obs):
    m, log_s=encoder(obs).chunk(2,dim=-1)
    return m, log_s.clamp(-5,2)
def elbo(obs, n_samples=8):
    m, log_s=posterior_params(obs)
    x=m+log_s.exp()*torch.randn(n_samples,*m.shape)
    reconstruction=torch.distributions.Normal(decoder(x),sigma).log_prob(obs).sum(-1).mean(0)
    kl=.5*(m.square()+torch.exp(2*log_s)-1-2*log_s).sum(-1)
    return (reconstruction-kl).mean()
for p in decoder.parameters(): p.requires_grad_(False)
opt=torch.optim.Adam(encoder.parameters(),lr=.005)
for _ in range(500):
    obs=Y[torch.randint(len(Y),(128,))]
    loss=-elbo(obs); opt.zero_grad(); loss.backward(); opt.step()

In [ ]:
def compare_posterior(index):
    with torch.no_grad():
        obs=Y[index:index+1]; g=torch.linspace(-6,6,1001)[:,None]
        lj=torch.distributions.Normal(decoder(g),sigma).log_prob(obs).sum(-1)+torch.distributions.Normal(0.,1.).log_prob(g[:,0])
        log_z=torch.logsumexp(lj,0)+torch.log(g[1,0]-g[0,0])
        m, log_s=posterior_params(obs)
        q=torch.distributions.Normal(m.squeeze(),log_s.exp().squeeze())
        plt.plot(g[:,0],torch.exp(lj-log_z),label='model posterior (grid)')
        plt.plot(g[:,0],q.log_prob(g[:,0]).exp(),label='q'); plt.legend(); plt.show()
        print('log evidence:', log_z.item(), 'MC ELBO:',elbo(obs,10000).item())
compare_posterior(0)
compare_posterior(12)

固定生成模型时，最大化 ELBO 等价于最小化 q 与真实后验之间的 KL 散度。高斯 q 不一定能表示真实后验的多个峰；编码器也不一定能对每个样本达到最优。蒙特卡洛估计的 ELBO 可能因采样误差偶尔超过数值似然。

In [ ]:
# VAE: 从头联合训练，不使用网格积分作为训练目标。
decoder=make_decoder()
encoder=nn.Sequential(nn.Linear(2,32),nn.Tanh(),nn.Linear(32,2))
opt=torch.optim.Adam(list(decoder.parameters())+list(encoder.parameters()),lr=.005)
history=[]
for _ in range(1000):
    obs=Y[torch.randint(len(Y),(128,))]
    loss=-elbo(obs); opt.zero_grad(); loss.backward(); opt.step(); history.append(-loss.item())
plt.plot(history); plt.ylabel('MC ELBO'); plt.show()
compare_posterior(0)
with torch.no_grad():
    compare((decoder(torch.randn(len(y),1))+sigma*torch.randn(len(y),2)).numpy())

改变隐变量维度将需要同时调整先验、编码器和解码器；训练时的采样仍可扩展，不需要指数增长的网格。VAE 的生成质量受模型、后验近似和优化影响，不保证拟合优于直接 MLE。